In [ ]:
from data_utils import *
import numpy as np
from tqdm import tqdm
import torch
from utils import getBasicScores,getFairnessScores,area_curve_metric

In [2]:
DATASET = 'beauty'

In [3]:
DATA_PATH = '../data/'
POS_PATH = f'top-preds/{DATASET}/stage-1-POS-only-EXPLS/'
NEG_PATH = f'top-preds/{DATASET}/stage-1-NEG-only-EXPLS'
ZERO_PATH = f'top-preds/{DATASET}/stage-1-ZERO-EXPLS/'

In [4]:
BATCH_SIZE=100

In [5]:
np.random.seed(999)

# Explanation Loading

In [6]:
pos_expls = load_pickle(os.path.join(POS_PATH,f'DEEPFM-{DATASET}-preds.pkl'))

In [7]:
neg_expls = load_pickle(os.path.join(NEG_PATH,f'DEEPFM-{DATASET}-preds.pkl'))

In [8]:
zero_expls = load_pickle(os.path.join(ZERO_PATH,f'DEEPFM-{DATASET}-preds.pkl'))

In [9]:
def map_sorted_preds_to_ranked_items(preds_file):
    ui_scores = preds_file['ui_scores']
    preds = preds_file['preds']
    
    assert isinstance(ui_scores, dict)
    assert isinstance(preds, list) or isinstance(preds, np.ndarray)

    mapped_results = {}
    user_ids = list(ui_scores.keys())
    num_candidates = len(next(iter(ui_scores.values())))  # assume uniform candidate size

    for idx, user_id in tqdm(enumerate(user_ids)):
        user_pred_scores = np.array(preds[idx * num_candidates : (idx + 1) * num_candidates])
        item_rank_dict = ui_scores[user_id]  # {item_id: -rank}
        
        # Invert to get: {-1: item_id1, -2: item_id2, ..., -100: item_id100}
        rank_to_item = {rank: item for item, rank in item_rank_dict.items()}
        
        # Sort predictions in descending order
        sorted_scores = np.sort(user_pred_scores)[::-1]
        
        # Map highest score to -1, second to -2, ..., lowest to -num_candidates
        user_item_score_map = {}
        for i in range(1, num_candidates + 1):
            rank = -i
            item_id = rank_to_item[rank]
            score = sorted_scores[i - 1]
            user_item_score_map[item_id] = (score,rank)
        
        mapped_results[user_id] = user_item_score_map

    return mapped_results

In [10]:
def get_score(u,v,mapping):
    return round(mapping[u][v][0],5),mapping[u][v][1]

In [11]:
def get_top_k(mapping,user, k):
    lst = sorted(mapping[user].items(), key=lambda x:x[1][0],reverse=True)
    if k == -1:
        k = len(lst)
    return [x[0] for x in lst[:k]]

In [12]:
new_pos_expls = map_sorted_preds_to_ranked_items(pos_expls)
new_neg_expls = map_sorted_preds_to_ranked_items(neg_expls)
new_zero_expls = map_sorted_preds_to_ranked_items(zero_expls)

22363it [00:01, 19758.34it/s]
22363it [00:00, 24884.62it/s]
22363it [00:00, 25512.81it/s]


In [13]:
targetItems = readTargetItem(os.path.join(DATA_PATH,DATASET,"targetItems.txt"))
datamaps = load_json(os.path.join(DATA_PATH,DATASET,"datamaps.json"))
targetItems = [int(datamaps["item2id"][x]) for x in targetItems]
targetItems[:5]

[271, 3983, 1979, 7928, 3649]

# Niche Item Selection

In [14]:
nicheItems = [int(x) for x in datamaps['item2id'].values() if int(x) not in targetItems]
print(f"Number of niche items: {len(nicheItems)}")
nicheItems[:5]

Number of niche items: 10891


[1, 2, 3, 4, 5]

In [15]:
RATIO = 0.01
num_items = int(RATIO * len(datamaps['item2id']))
num_items

121

In [16]:
nicheItems = np.random.choice(nicheItems,size=num_items,replace=False)
assert len(nicheItems) == len(set(nicheItems))
nicheItems[:5]

array([11080,  3305,  3185,  8311,  3207])

# Re-ranking using this algorithm

In [17]:
def rerank_with_niche_boost(user, new_pos_expls, new_zero_expls, nicheItems):
    """
    Re-rank using zero explanation scores for all items,
    except for niche items where we use the positive explanation scores.

    Parameters:
    - user: the current user to check
    - new_pos_expls: dict {user: {item: (score, -rank)}}
    - new_zero_expls: dict {user: {item: (score, -rank)}}
    - nicheItems: set of item IDs

    Returns:
    - result: dict {user: {item: score}}
    """
    reranked_scores = {}

    for item in new_zero_expls[user]:
        if item in nicheItems:
            score,_ = get_score(user, item, new_pos_expls)
            reranked_scores[item] = score
        else:
            score,_ = get_score(user, item, new_zero_expls)
            reranked_scores[item] = score

    return reranked_scores

In [18]:
all_info = []
golds,preds = [],[]
niche_golds = []

ui_scores = dict()
gt = dict()
niche_gt = dict()
top = [1,2,3,5,10,20]

users = list(new_pos_expls.keys())
for stepv, user in tqdm(enumerate(users)):
    user = int(user)
    gold_item = int(zero_expls['gt'][user][0])
    scores_dict = rerank_with_niche_boost(user, new_pos_expls, new_zero_expls, nicheItems)
    rerank_lst = sorted(scores_dict.items(),key = lambda x:x[1], reverse=True)
    gt[user] = [gold_item]
    niche_gt[user] = nicheItems.tolist() # we only check niche item relevance!
    pred_dict = {}

    for j in range(len(rerank_lst)):

        item, score = rerank_lst[j]
        pred_dict[item] = -(j + 1)
        label = int(gold_item == item)
        niche_label = int(item in nicheItems)
        golds.append(label)
        niche_golds.append(niche_label)
        preds.append(score)
    
    ui_scores[user] = pred_dict

print("# golds: ",len(golds))
print("# niche golds: ",len(niche_golds))
print("# preds: ",len(preds))
print(f"# niche items overall across all users: {sum(niche_golds)}")
print("Original Recommendation Performance")
_, Recommendresults = getBasicScores(ui_scores, gt, top)
print("\nOriginal AUC: ",area_curve_metric(golds,preds)) 
print("Original Fairness Performance")
FairResults = getFairnessScores(ui_scores, targetItems, top, len(datamaps['item2id']))

22363it [00:16, 1339.70it/s]


# golds:  2236300
# niche golds:  2236300
# preds:  2236300
# niche items overall across all users: 22323
Original Recommendation Performance

NDCG@1	Rec@1	Hits@1	Prec@1	MAP@1	MRR@1
0.1676	0.1676	0.1676	0.1676	0.1676	0.1676

NDCG@2	Rec@2	Hits@2	Prec@2	MAP@2	MRR@2
0.1698	0.1712	0.1712	0.0856	0.1694	0.1694

NDCG@3	Rec@3	Hits@3	Prec@3	MAP@3	MRR@3
0.1723	0.1760	0.1760	0.0587	0.1710	0.1710

NDCG@5	Rec@5	Hits@5	Prec@5	MAP@5	MRR@5
0.1774	0.1886	0.1886	0.0377	0.1738	0.1738

NDCG@10	Rec@10	Hits@10	Prec@10	MAP@10	MRR@10
0.1912	0.2322	0.2322	0.0232	0.1793	0.1793

NDCG@20	Rec@20	Hits@20	Prec@20	MAP@20	MRR@20
0.2172	0.3363	0.3363	0.0168	0.1863	0.1863

Original AUC:  0.5767732021987177
Original Fairness Performance

PR@1	LTR@1	KLD@1	Gini@1	SDI@1	UHC@1
0.4442	0.5558	0.3945	0.4931	0.4938	0.4442

PR@2	LTR@2	KLD@2	Gini@2	SDI@2	UHC@2
0.4116	0.5884	0.3323	0.5373	0.4844	0.6264

PR@3	LTR@3	KLD@3	Gini@3	SDI@3	UHC@3
0.3962	0.6038	0.3045	0.5549	0.4785	0.7335

PR@5	LTR@5	KLD@5	Gini@5	SDI@5	UHC@5
0.3754	0.6246	0

In [19]:
print("Positive Explanation Ablation: \nRecommendation Performance wrt Niche Items")
_, Recommendresults = getBasicScores(ui_scores, niche_gt, top)
print("\nNiche Item AUC: ",area_curve_metric(niche_golds,preds))

Positive Explanation Ablation: 
Recommendation Performance wrt Niche Items

NDCG@1	Rec@1	Hits@1	Prec@1	MAP@1	MRR@1
0.0114	0.0001	0.0114	0.0114	0.0114	0.0114

NDCG@2	Rec@2	Hits@2	Prec@2	MAP@2	MRR@2
0.0208	0.0002	0.0262	0.0136	0.0188	0.0188

NDCG@3	Rec@3	Hits@3	Prec@3	MAP@3	MRR@3
0.0295	0.0004	0.0438	0.0154	0.0247	0.0246

NDCG@5	Rec@5	Hits@5	Prec@5	MAP@5	MRR@5
0.0457	0.0007	0.0831	0.0180	0.0334	0.0335

NDCG@10	Rec@10	Hits@10	Prec@10	MAP@10	MRR@10
0.0820	0.0019	0.1945	0.0230	0.0478	0.0479

NDCG@20	Rec@20	Hits@20	Prec@20	MAP@20	MRR@20
0.1343	0.0043	0.3944	0.0261	0.0606	0.0614

Niche Item AUC:  0.7826383187753574


In [20]:
print("Zero Score for all: \nRecommendation Performance wrt Niche Items")
_, Recommendresults = getBasicScores(zero_expls['ui_scores'], niche_gt, top)
print("\nNiche Item AUC: ",area_curve_metric(niche_golds,zero_expls['preds'])) 

Zero Score for all: 
Recommendation Performance wrt Niche Items

NDCG@1	Rec@1	Hits@1	Prec@1	MAP@1	MRR@1
0.0067	0.0001	0.0067	0.0067	0.0067	0.0067

NDCG@2	Rec@2	Hits@2	Prec@2	MAP@2	MRR@2
0.0116	0.0001	0.0144	0.0072	0.0106	0.0106

NDCG@3	Rec@3	Hits@3	Prec@3	MAP@3	MRR@3
0.0158	0.0002	0.0229	0.0077	0.0134	0.0134

NDCG@5	Rec@5	Hits@5	Prec@5	MAP@5	MRR@5
0.0221	0.0003	0.0383	0.0078	0.0168	0.0168

NDCG@10	Rec@10	Hits@10	Prec@10	MAP@10	MRR@10
0.0355	0.0007	0.0803	0.0083	0.0221	0.0222

NDCG@20	Rec@20	Hits@20	Prec@20	MAP@20	MRR@20
0.0563	0.0015	0.1635	0.0088	0.0274	0.0278

Niche Item AUC:  0.5007752570859337


In [21]:
def compute_avg_rank_score(ablation_score_map,nicheItems,K):
    print(f"K:{K}")
    niche_zero_score = []
    niche_pos_score = []
    niche_zero_rank = []
    niche_pos_rank = []
    niche_diff_score = []
    niche_diff_rank = []
    
    for user in ablation_score_map:
        item_maps = ablation_score_map[user]
        
        item_list = sorted(item_maps.items(),key = lambda x:x[1][0],reverse=True)
        item_list = [x[0] for x in item_list][:K]
        for item in item_list:
            if item in nicheItems:
                pos_score, pos_rank = item_maps[item]
                zero_score, zero_rank = get_score(user,item, new_zero_expls)
                zero_rank = int(-zero_rank)
                pos_rank = int(-pos_rank)
                niche_zero_score.append(zero_score)
                niche_pos_score.append(pos_score)
                niche_zero_rank.append(zero_rank)
                niche_pos_rank.append(pos_rank)
                # niche_diff_rank.append(zero_rank - pos_rank)
                # niche_diff_score.append(pos_score-zero_score)

    print("\tlen(niche_pos_score): ",len(niche_pos_score))
    print("\tlen(niche_zero_score): ",len(niche_zero_score))
    print('\n')
    DEN = len(ablation_score_map) * K
    print(f"\tE(niche pos score): {np.mean(niche_pos_score):.6f}")
    print(f"\tE(niche zero score): {np.mean(niche_zero_score):.6f}")
    print(f"\tDiff between E(niche_pos_score) - E(niche_zero_score): {np.mean(niche_pos_score) - np.mean(niche_zero_score):.6f}")
    # print(f"E(diff between score):{np.mean(niche_diff_score):.6f}")
    print('\n')
    print(f"\tE(niche pos rank): {np.mean(niche_pos_rank):.6f}")
    print(f"\tE(niche zero rank): {np.mean(niche_zero_rank):.6f}")
    print(f"\tDiff between E(niche_zero_rank) - E(niche_pos_rank): {np.mean(niche_zero_rank) - np.mean(niche_pos_rank):.6f}")
    # print(f"E(diff between ranks): {np.mean(niche_diff_rank):.6f}")
    print('='*50)
    return

In [22]:
ablation_score_map = map_sorted_preds_to_ranked_items({'ui_scores':ui_scores,'gt':gt, 'golds':golds, 'preds':preds})

22363it [00:00, 24267.43it/s]


In [23]:
for K in [1,2,3,5,10,20]:
    compute_avg_rank_score(ablation_score_map,nicheItems,K)

K:1
	len(niche_pos_score):  254
	len(niche_zero_score):  254


	E(niche pos score): 0.998495
	E(niche zero score): 0.742468
	Diff between E(niche_pos_score) - E(niche_zero_score): 0.256027


	E(niche pos rank): 1.000000
	E(niche zero rank): 10.228346
	Diff between E(niche_zero_rank) - E(niche_pos_rank): 9.228346
K:2
	len(niche_pos_score):  607
	len(niche_zero_score):  607


	E(niche pos score): 0.997090
	E(niche zero score): 0.675414
	Diff between E(niche_pos_score) - E(niche_zero_score): 0.321676


	E(niche pos rank): 1.581549
	E(niche zero rank): 13.703460
	Diff between E(niche_zero_rank) - E(niche_pos_rank): 12.121911
K:3
	len(niche_pos_score):  1030
	len(niche_zero_score):  1030


	E(niche pos score): 0.995424
	E(niche zero score): 0.666874
	Diff between E(niche_pos_score) - E(niche_zero_score): 0.328550


	E(niche pos rank): 2.164078
	E(niche zero rank): 15.470874
	Diff between E(niche_zero_rank) - E(niche_pos_rank): 13.306796
K:5
	len(niche_pos_score):  2015
	len(niche_zero_score

In [14]:
for K in [100]:
    compute_avg_rank_score(ablation_score_map,nicheItems,K)

K:100
	len(niche_pos_score):  22323
	len(niche_zero_score):  22323


	E(niche pos score): 0.934798
	E(niche zero score): 0.433036
	Diff between E(niche_pos_score) - E(niche_zero_score): 0.501762


	E(niche pos rank): 22.512655
	E(niche zero rank): 52.057564
	Diff between E(niche_zero_rank) - E(niche_pos_rank): 29.544909


In [24]:
OUTPUT_DIR = os.path.join("top-preds","POS-ABL")
os.makedirs(OUTPUT_DIR,exist_ok=True)

In [25]:
save_path = os.path.join(OUTPUT_DIR,f"DEEPFM-{DATASET}-preds.pkl")
save_pickle({'ui_scores':ui_scores,
             'gt':gt, 'golds':golds, 
             'preds':preds, 'niche_golds':niche_golds,
             'niche_gt':niche_gt
            },
            save_path)

In [13]:
curr_preds = load_pickle(os.path.join("top-preds",DATASET,"POS-ABL",f"DEEPFM-{DATASET}-preds.pkl"))
curr_preds.keys()

dict_keys(['ui_scores', 'gt', 'golds', 'preds', 'niche_golds', 'niche_gt'])

In [14]:
ablation_score_map = map_sorted_preds_to_ranked_items(curr_preds)

22363it [00:00, 28051.27it/s]


In [15]:
nicheItems = set()
for key in curr_preds['niche_gt']:
    nicheItems = nicheItems.union(set(curr_preds['niche_gt'][key]))
len(nicheItems)

121

In [16]:
def compute_avg_MIR(ablation_score_map,nicheItems,K):
    print(f"K:{K}")
    niche_zero_rank = []
    niche_pos_rank = []
    
    for user in ablation_score_map:
        item_maps = ablation_score_map[user]
        
        item_list = sorted(item_maps.items(),key = lambda x:x[1][0],reverse=True)
        item_list = [x[0] for x in item_list][:K]
        for item in item_list:
            if item in nicheItems:
                _, pos_rank = item_maps[item]
                _, zero_rank = get_score(user,item, new_zero_expls)
                zero_rank = int(-zero_rank)
                pos_rank = int(-pos_rank)
                niche_zero_rank.append(1/zero_rank)
                niche_pos_rank.append(1/pos_rank)

    DEN = len(ablation_score_map) * K
    print(f"\tE(niche pos IR): {np.mean(niche_pos_rank):.6f}")
    print(f"\tE(niche zero IR): {np.mean(niche_zero_rank):.6f}")
    print(f"\tDiff between E(niche_pos_IR) - E(niche_zero_IR): {-np.mean(niche_zero_rank) + np.mean(niche_pos_rank):.6f}")
    # print(f"E(diff between ranks): {np.mean(niche_diff_rank):.6f}")
    print('='*50)
    return

In [17]:
for K in [100]:
    compute_avg_MIR(ablation_score_map,nicheItems,K)

K:100
	E(niche pos IR): 0.088269
	E(niche zero IR): 0.045122
	Diff between E(niche_pos_IR) - E(niche_zero_IR): 0.043147
